<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/%EB%8F%84%EA%B5%AC_3%EA%B0%9C_%ED%8E%91%EC%85%98%EC%BD%9C%EB%A7%81_%EC%97%B0%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip -q install langchain openai tiktoken yfinance langchain_community langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [5]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [16]:
from pydantic import BaseModel, Field
from typing import Optional, Type, List

from langchain_openai import ChatOpenAI
from langchain.tools import BaseTool

from datetime import datetime, timedelta

#도구 설정

1. 주가 반환 도구

In [11]:
import yfinance as yf

def get_stock_price(symbol):
    """
    주어진 주식 심볼의 최신 종가를 반환하는 함수

    :param symbol: 주식 심볼 (예: 'AAPL' for Apple Inc.)
    :return: 소수점 둘째 자리까지 반올림된 최신 종가
    """
    ticker = yf.Ticker(symbol)
    todays_data = ticker.history(period='1d')
    return round(todays_data['Close'].iloc[0], 2)

class StockPriceCheckInput(BaseModel):
    """Input for Stock price check."""

    stockticker: str = Field(..., description="Ticker symbol for stock or index")

class StockPriceTool(BaseTool):
    name: str = "get_stock_ticker_price"
    description: str = "주식 가격을 알아야 할 때 유용합니다. yfinance API에서 사용되는 주식 티커(종목 코드)를 입력해야 합니다."

    def _run(self, stockticker: str):
        # print("i'm running")
        price_response = get_stock_price(stockticker)
        return price_response

    def _arun(self, stockticker: str):
        raise NotImplementedError("This tool does not support async")

    args_schema: Optional[Type[BaseModel]] = StockPriceCheckInput

2. 주가 변동률 계산 도구

In [12]:
def calculate_performance(symbol, days_ago):
    """
    주어진 기간 동안 특정 주식의 성과(가격 변동률)를 계산하는 함수.

    :param symbol: 주식 심볼 (예: 'AAPL' for Apple Inc.)
    :param days_ago: 몇 일 전부터의 성과를 계산할지 지정하는 정수
    :return: 주식의 가격 변동률 (백분율)
    """
    # 주어진 심볼에 대한 Ticker 객체 생성
    ticker = yf.Ticker(symbol)

    # 현재 날짜를 종료 날짜로 설정
    end_date = datetime.now()

    # 시작 날짜 계산 (현재로부터 days_ago일 전)
    start_date = end_date - timedelta(days=days_ago)

    # 날짜를 'YYYY-MM-DD' 형식의 문자열로 변환
    start_date = start_date.strftime('%Y-%m-%d')
    end_date = end_date.strftime('%Y-%m-%d')

    # 지정된 기간의 주가 데이터 가져오기
    historical_data = ticker.history(start=start_date, end=end_date)

    # 시작 날짜의 종가와 마지막 날짜의 종가 가져오기
    old_price = historical_data['Close'].iloc[0]
    new_price = historical_data['Close'].iloc[-1]

    # 퍼센트 변화율 계산
    # ((새 가격 - 옛 가격) / 옛 가격) * 100
    percent_change = ((new_price - old_price) / old_price) * 100

    # 결과를 소수점 둘째 자리까지 반올림하여 반환
    return round(percent_change, 2)

In [13]:
class StockChangePercentageCheckInput(BaseModel):
    """주식 가격 변동률 확인을 위한 입력 모델"""

    stockticker: str = Field(..., description="주식 또는 지수의 티커 심볼")
    days_ago: int = Field(..., description="몇 일 전부터 확인할지 지정하는 정수")

class StockPercentageChangeTool(BaseTool):
    """주식 가격 변동률을 계산하는 도구"""

    name: str = "get_price_change_percent"
    description: str = "주식 가치의 백분율 변화를 확인해야 할 때 유용합니다. yfinance API에서 사용되는 주식 티커와 변화를 확인할 일수를 입력해야 합니다."

    def _run(self, stockticker: str, days_ago: int) -> float:
        """
        주어진 주식의 가격 변동률을 계산합니다.
        :param stockticker: 주식 티커 심볼
        :param days_ago: 몇 일 전부터 계산할지 지정하는 정수
        :return: 가격 변동률
        """
        price_change_response = calculate_performance(stockticker, days_ago)
        return price_change_response

    def _arun(self, stockticker: str, days_ago: int):
        """비동기 실행을 지원하지 않음을 나타냅니다."""
        raise NotImplementedError("이 도구는 비동기를 지원하지 않습니다")

    args_schema: Optional[Type[BaseModel]] = StockChangePercentageCheckInput

3. 최고 성과 주식 판별 도구

In [17]:
def get_best_performing(stocks, days_ago):
    """
    주어진 주식 목록에서 지정된 기간 동안 가장 좋은 성과를 보인 주식을 찾는 함수.

    :param stocks: 주식 심볼 리스트 (예: ['AAPL', 'GOOGL', 'MSFT'])
    :param days_ago: 몇 일 전부터의 성과를 계산할지 지정하는 정수
    :return: 튜플 (최고 성과 주식 심볼, 해당 주식의 성과율)
    """
    best_stock = None  # 최고 성과 주식을 저장할 변수
    best_performance = None  # 최고 성과율을 저장할 변수

    # 주어진 주식 목록을 순회
    for stock in stocks:
        try:
            # 각 주식의 성과 계산
            performance = calculate_performance(stock, days_ago)

            # 현재 주식의 성과가 지금까지의 최고 성과보다 좋으면 업데이트
            if best_performance is None or performance > best_performance:
                best_stock = stock
                best_performance = performance
        except Exception as e:
            # 성과 계산 중 오류 발생 시 오류 메시지 출력
            print(f"Could not calculate performance for {stock}: {e}")

    # 최고 성과 주식과 그 성과율 반환
    return best_stock, best_performance

In [18]:
class StockBestPerformingInput(BaseModel):
    """최고 성과 주식 찾기를 위한 입력 모델"""

    stocktickers: List[str] = Field(..., description="주식 또는 지수의 티커 심볼 리스트")
    days_ago: int = Field(..., description="몇 일 전부터 확인할지 지정하는 정수")

class StockGetBestPerformingTool(BaseTool):
    """여러 주식 중 최고 성과 주식을 찾는 도구"""

    name: str = "get_best_performing"
    description: str = "특정 기간 동안 여러 주식의 성과를 확인해야 할 때 유용합니다. yfinance API에서 사용되는 주식 티커 리스트와 변화를 확인할 일수를 입력해야 합니다."

    def _run(self, stocktickers: List[str], days_ago: int) -> tuple:
        """
        주어진 주식 리스트에서 최고 성과 주식을 찾습니다.
        :param stocktickers: 주식 티커 심볼 리스트
        :param days_ago: 몇 일 전부터 계산할지 지정하는 정수
        :return: 최고 성과 주식과 그 성과율
        """
        price_change_response = get_best_performing(stocktickers, days_ago)
        return price_change_response

    def _arun(self, stockticker: List[str], days_ago: int):
        """비동기 실행을 지원하지 않음을 나타냅니다."""
        raise NotImplementedError("이 도구는 비동기를 지원하지 않습니다")

    args_schema: Optional[Type[BaseModel]] = StockBestPerformingInput

#LLM 에이전트 호출

In [40]:
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor, initialize_agent, AgentType
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

tools = [StockPriceTool(), StockPercentageChangeTool(), StockGetBestPerformingTool()]
llm = ChatOpenAI(temperature=0, model="gpt-4o")
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 주식 시장 분석 전문가입니다. 주어진 도구를 활용해 답변하세요."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent = agent, tools =  tools)

agent_executor.invoke({'input' : '구글 주식 가격'})

ag = initialize_agent(tools, llm, agent = AgentType.OPENAI_FUNCTIONS, verbose=True)
ag.run('구글 주식')

{'input': '구글 주식 가격', 'output': '현재 구글(GOOGL) 주식 가격은 $307.04입니다.'}

In [43]:
# 1. 최신 LangGraph 및 핵심 모듈 임포트
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# 2. 도구 및 LLM 셋업 (기존과 동일)
tools = [StockPriceTool(), StockPercentageChangeTool(), StockGetBestPerformingTool()]
llm = ChatOpenAI(temperature=0, model="gpt-4o")

# 3. 시스템 프롬프트 정의
# MessagesPlaceholder나 복잡한 템플릿 없이 단순 문자열로 시스템 역할을 부여합니다.
system_prompt = "당신은 주식 시장 분석 전문가입니다. 주어진 도구를 활용해 답변하세요."

# 4. 에이전트 생성 (AgentExecutor, initialize_agent를 모두 대체)
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt # 시스템 프롬프트를 여기에 바로 넣습니다.
)

# 5. 에이전트 실행 (run 대신 invoke 사용, 메시지 리스트 형태로 전달)
result = agent.invoke({
    "messages": [("user", "구글 주식 가격 알려줘")]
})

# 6. 최종 결과 출력 (결과물은 메시지 리스트의 가장 마지막 요소에 담깁니다)
print(result["messages"][-1].content)

/tmp/ipykernel_3663/1977964812.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


현재 구글(GOOGL) 주식 가격은 $307.04입니다.


In [44]:
result = agent.invoke({
    "messages": [("user", "구글, 한화, 삼성 중 1년전에 비해 가장 성장률이 높은 주식은?")]
})

print(result["messages"][-1].content)

1년 전에 비해 가장 높은 성장률을 보인 주식은 삼성(005930.KS)으로, 258.12%의 성장률을 기록했습니다.


In [45]:
result = agent.invoke({
    "messages": [("user", "구글, 한화, 삼성 중 1년전에 비해 가장 성장률이 높은 주식은? 다른 주식들은 얼마나 성장했는지도 같이 말해줘")]
})

print(result["messages"][-1].content)

1년 전과 비교했을 때, 삼성의 주식이 258.12%로 가장 높은 성장률을 기록했습니다. 다른 주식들의 성장률은 다음과 같습니다:

- 구글(GOOGL): 87.82%
- 한화(000880.KS): 147.06% 

삼성이 가장 높은 성장률을 보였으며, 한화와 구글이 그 뒤를 잇고 있습니다.


In [46]:
result = agent.invoke({
    "messages": [("user", "구글, 한화, 삼성 중 1년전에 비해 가장 성장률이 높은 주식은? 다른 주식들은 얼마나 성장했는지도 같이 말해줘")]
})

print(result["messages"][-1].content)

1년 전과 비교했을 때, 삼성의 주식이 258.12%로 가장 높은 성장률을 보였습니다. 다른 주식들의 성장률은 다음과 같습니다:

- 구글(GOOGL): 87.82%
- 한화(000880.KS): 147.06%


In [52]:
result = agent.invoke({
    "messages": [("user", "랜덤 아무 주식 3개의 현황. 그 중에서 이번 분기 제일 잘나가는 주식은? 그 외의 주식은 어떤가?")]
})

print(result["messages"][-1].content)

이번 분기 동안 세 개의 주식(AAPL, GOOGL, AMZN)의 성과는 다음과 같습니다:

1. **GOOGL (Alphabet Inc.)**: -1.66%로 가장 적은 하락을 보이며, 이번 분기에서 가장 잘 나가는 주식입니다.
2. **AAPL (Apple Inc.)**: -6.1% 하락했습니다.
3. **AMZN (Amazon.com Inc.)**: -6.93% 하락했습니다.

GOOGL이 상대적으로 가장 적은 하락을 보이며, 다른 두 주식보다 더 나은 성과를 보였습니다. AAPL과 AMZN은 비슷한 수준의 하락을 보였지만, AMZN이 약간 더 큰 하락을 기록했습니다.
